In [79]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [80]:
df = pd.read_csv("../data/cleaned/news_1.csv")

## Data preprocessing

### target class balance

In [81]:
(df["target"].value_counts()/len(df["target"])) * 100        # the classes are balanced

target
0    52.298543
1    47.701457
Name: count, dtype: float64

In [82]:
df.isnull().sum()

title      0
text       0
subject    0
target     0
dtype: int64

### duplicates dropped

In [83]:
df = df.drop_duplicates()

### strip, lowercase, and excess spaces

In [84]:
import re
df[["title", "text", "subject"]] = df[["title", "text", "subject"]].apply(lambda col: col.str.strip().str.lower().str.replace(r"\s+", " ", regex=True))

In [85]:
df["subject"].unique()

array(['politics', 'worldnews', 'politicsnews', 'news', 'government news',
       'left-news', 'us_news', 'middle-east'], dtype=object)

In [86]:
df.loc[df["subject"] == "news", "subject"] = "unknown"

### Analysing Text columns

In [69]:
samples = df["text"].sample(5)
for sample in samples:
    print(sample, "\n")

washington (reuters) - house speaker paul ryan on wednesday said a range of options to provide funds to fight zika, adding that lawmakers take the threat seriously but have not yet decided the best way to allocate resources to prevent and combat the deadly virus. “we’re looking at all different options,” adding that the white house has begun providing congressional staff with answers to questions over president barack obama’s funding request. “the administration has a bit of a track record of over-requesting what they need.” 

brussels (reuters) - european union chief executive jean-claude juncker said on friday that brexit talks would move on to the second phase to talk about trade after he judged that sufficient progress had been made on the divorce deal. the commission has just formally decided to recommend to the european council that sufficient progress has now been made on the strict terms of the divorce, he told an early morning press conference with british prime minister there

## NOTE: I am skipping other text preprocessing like removing html tags, puncations, numbers, urls, stopwords, stemming/lemmatization I want to check iF they effect or not

In [87]:
X = df.drop("target", axis=1)
y = df["target"]

In [88]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y ,random_state=42)

In [89]:
preprocessor = ColumnTransformer(
    transformers=[
        ("title_bow", CountVectorizer(), "title"),
        ("text_bow", CountVectorizer(), "text"),
        ("subject_ohe", OneHotEncoder(handle_unknown="ignore"), ["subject"]),
    ]
)

In [91]:
model = Pipeline([
    ("features", preprocessor),
    ("clf", MultinomialNB())
])

In [92]:
model.fit(X_train, y_train)

,steps,"[('features', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('title_bow', ...), ('text_bow', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [93]:
y_pred = model.predict(X_test)

In [94]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9714669352131587

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      4695
           1       0.97      0.97      0.97      4242

    accuracy                           0.97      8937
   macro avg       0.97      0.97      0.97      8937
weighted avg       0.97      0.97      0.97      8937


Confusion Matrix:

[[4566  129]
 [ 126 4116]]
